# 04 — Hybrid Model & Evaluation
Combines collaborative filtering (SVD), content-based filtering (SBERT), and popularity.

**Formula:** Final Score = α × CF Score + β × Content Score + γ × Popularity Score

Evaluates using Precision@K and NDCG@K on the held-out test set.

Run notebooks 01, 02, and 03 first.

## 1. Load All Artifacts

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Load data
train   = pd.read_csv('../data/processed/train.csv')
test    = pd.read_csv('../data/processed/test.csv')
catalog = pd.read_csv('../data/processed/item_catalog.csv')

# Load content-based artifacts
with open('../data/processed/content_data.pkl', 'rb') as f:
    content_data = pickle.load(f)

sbert_emb      = content_data['sbert_embeddings']
cb_item_to_idx = content_data['item_to_idx']

# Load collaborative filtering artifacts (scipy SVD)
with open('../data/processed/cf_data.pkl', 'rb') as f:
    cf_data = pickle.load(f)

user_factors    = cf_data['user_factors']     # (n_users, n_components)
item_factors    = cf_data['item_factors']     # (n_items, n_components)
user_to_idx     = cf_data['user_to_idx']
item_to_idx     = cf_data['item_to_idx']
idx_to_item     = cf_data['idx_to_item']
user_seen_items = cf_data['user_seen_items']
all_item_ids    = cf_data['all_item_ids']

# Normalize popularity to [0, 1]
catalog['popularity_norm'] = catalog['popularity'] / catalog['popularity'].max()
pop_map = dict(zip(catalog['item_id'], catalog['popularity_norm']))

print(f'Train: {len(train):,} | Test: {len(test):,} | Catalog: {len(catalog):,}')
print(f'Known users: {len(user_to_idx):,} | SVD factors shape: {user_factors.shape}')

## 2. Score Functions

In [ ]:
def get_cf_scores(user_id, candidate_items):
    """
    Score candidates using SVD dot-product (user vector · item factors).
    Returns dict: {item_id: normalized_score}
    """
    if user_id not in user_to_idx:
        return {item: 0.0 for item in candidate_items}  # cold start

    u_idx      = user_to_idx[user_id]
    u_vector   = user_factors[u_idx]  # (n_components,)

    cand_items = [iid for iid in candidate_items if iid in item_to_idx]
    cand_idxs  = np.array([item_to_idx[iid] for iid in cand_items])

    if len(cand_idxs) == 0:
        return {item: 0.0 for item in candidate_items}

    # Dot product scoring
    raw    = item_factors[cand_idxs] @ u_vector
    min_s, max_s = raw.min(), raw.max()
    rng    = max_s - min_s if max_s != min_s else 1.0
    normed = (raw - min_s) / rng

    score_map = dict(zip(cand_items, normed.tolist()))
    return {item: score_map.get(item, 0.0) for item in candidate_items}


def get_content_scores(user_id, candidate_items, train_df):
    """
    Score candidates using SBERT user profile vector.
    Profile = weighted average of embeddings of items user interacted with.
    Returns dict: {item_id: cosine_similarity_score}
    """
    user_history = train_df[train_df['user_id'] == user_id]

    if user_history.empty:
        return {item: 0.0 for item in candidate_items}

    profile  = np.zeros(sbert_emb.shape[1])
    total_w  = 0.0

    for _, row in user_history.iterrows():
        if row['item_id'] in cb_item_to_idx:
            idx      = cb_item_to_idx[row['item_id']]
            profile += sbert_emb[idx] * row['interaction_weight']
            total_w += row['interaction_weight']

    if total_w == 0:
        return {item: 0.0 for item in candidate_items}

    profile = (profile / total_w).reshape(1, -1)

    scores = {}
    for item_id in candidate_items:
        if item_id in cb_item_to_idx:
            idx = cb_item_to_idx[item_id]
            scores[item_id] = float(cosine_similarity(profile, sbert_emb[idx].reshape(1, -1))[0][0])
        else:
            scores[item_id] = 0.0

    return scores


print('Score functions ready.')

## 3. Hybrid Recommender

In [ ]:
def hybrid_recommend(user_id, train_df, top_n=10, alpha=0.5, beta=0.3, gamma=0.2):
    """
    Final Score = α × CF Score + β × Content Score + γ × Popularity Score
    Weights must sum to 1.0.
    """
    assert abs(alpha + beta + gamma - 1.0) < 1e-6, 'Weights must sum to 1.0'

    already_seen = set(train_df[train_df['user_id'] == user_id]['item_id'].unique())
    candidates   = [iid for iid in catalog['item_id'] if iid not in already_seen]

    cf_scores      = get_cf_scores(user_id, candidates)
    content_scores = get_content_scores(user_id, candidates, train_df)

    results = []
    for iid in candidates:
        cf_s  = cf_scores.get(iid, 0.0)
        cb_s  = content_scores.get(iid, 0.0)
        pop_s = pop_map.get(iid, 0.0)
        final = alpha * cf_s + beta * cb_s + gamma * pop_s
        results.append({
            'item_id':       iid,
            'final_score':   final,
            'cf_score':      cf_s,
            'content_score': cb_s,
            'pop_score':     pop_s
        })

    recs = pd.DataFrame(results).sort_values('final_score', ascending=False).head(top_n)
    recs = recs.merge(catalog[['item_id', 'item_title', 'avg_price']], on='item_id', how='left')

    for col in ['final_score', 'cf_score', 'content_score', 'pop_score']:
        recs[col] = recs[col].round(4)

    return recs.reset_index(drop=True)


# Test on a sample user
sample_user = train['user_id'].value_counts().index[0]
print(f'Hybrid recommendations for user: {sample_user}')
display(hybrid_recommend(sample_user, train, top_n=10))

## 4. Evaluation — Precision@K and NDCG@K

In [ ]:
def precision_at_k(recommended, relevant, k):
    top_k = recommended[:k]
    return len(set(top_k) & set(relevant)) / k


def ndcg_at_k(recommended, relevant, k):
    top_k    = recommended[:k]
    dcg      = sum(1 / np.log2(i + 2) for i, item in enumerate(top_k) if item in set(relevant))
    ideal    = min(len(relevant), k)
    idcg     = sum(1 / np.log2(i + 2) for i in range(ideal))
    return dcg / idcg if idcg > 0 else 0.0


K_VALUES     = [5, 10, 20]
N_EVAL_USERS = 200

train_users = set(train['user_id'].unique())
test_users  = set(test['user_id'].unique())
eval_users  = list(train_users & test_users)
np.random.seed(42)
eval_users  = np.random.choice(eval_users, size=min(N_EVAL_USERS, len(eval_users)), replace=False)

print(f'Evaluating on {len(eval_users)} users...')

results = {k: {'precision': [], 'ndcg': []} for k in K_VALUES}

for i, user_id in enumerate(eval_users):
    if i % 50 == 0:
        print(f'  {i}/{len(eval_users)}...')

    relevant = test[test['user_id'] == user_id]['item_id'].tolist()
    if not relevant:
        continue

    recs        = hybrid_recommend(user_id, train, top_n=max(K_VALUES))
    recommended = recs['item_id'].tolist()

    for k in K_VALUES:
        results[k]['precision'].append(precision_at_k(recommended, relevant, k))
        results[k]['ndcg'].append(ndcg_at_k(recommended, relevant, k))

print('\nEvaluation complete!')

In [ ]:
print('=' * 42)
print(f'{"K":>4}  {"Precision@K":>13}  {"NDCG@K":>10}')
print('-' * 42)
for k in K_VALUES:
    p = np.mean(results[k]['precision'])
    n = np.mean(results[k]['ndcg'])
    print(f'{k:>4}  {p:>13.4f}  {n:>10.4f}')
print('=' * 42)
print('\nNote: Scores above 0.05 Precision@10 are solid for implicit feedback datasets.')

## 5. Tune Alpha / Beta / Gamma Weights

In [ ]:
weight_combos = [
    (1.0, 0.0, 0.0),  # CF only
    (0.0, 1.0, 0.0),  # Content only
    (0.0, 0.0, 1.0),  # Popularity only
    (0.6, 0.3, 0.1),  # CF-heavy
    (0.3, 0.6, 0.1),  # Content-heavy
    (0.5, 0.3, 0.2),  # Balanced (default)
    (0.4, 0.4, 0.2),  # Equal CF + Content
]

K          = 10
tune_users = eval_users[:50]
tuning_results = []

for alpha, beta, gamma in weight_combos:
    p_scores, n_scores = [], []
    for user_id in tune_users:
        relevant = test[test['user_id'] == user_id]['item_id'].tolist()
        if not relevant:
            continue
        recs        = hybrid_recommend(user_id, train, top_n=K, alpha=alpha, beta=beta, gamma=gamma)
        recommended = recs['item_id'].tolist()
        p_scores.append(precision_at_k(recommended, relevant, K))
        n_scores.append(ndcg_at_k(recommended, relevant, K))

    tuning_results.append({
        'alpha': alpha, 'beta': beta, 'gamma': gamma,
        f'Precision@{K}': round(np.mean(p_scores), 4),
        f'NDCG@{K}':      round(np.mean(n_scores), 4),
    })
    print(f'α={alpha} β={beta} γ={gamma}  →  P@{K}={tuning_results[-1][f"Precision@{K}"]:.4f}  NDCG@{K}={tuning_results[-1][f"NDCG@{K}"]:.4f}')

tuning_df = pd.DataFrame(tuning_results).sort_values(f'NDCG@{K}', ascending=False)
print(f'\nBest weights by NDCG@{K}:')
display(tuning_df.head(3))

## 6. Final Demo

In [ ]:
best       = tuning_df.iloc[0]
BEST_ALPHA = best['alpha']
BEST_BETA  = best['beta']
BEST_GAMMA = best['gamma']

print(f'Best weights: α={BEST_ALPHA}, β={BEST_BETA}, γ={BEST_GAMMA}')
print()

for user_id in list(eval_users[:3]):
    print(f'=== Recommendations for user {user_id} ===')
    history = train[train['user_id'] == user_id]['item_title'].value_counts().head(3)
    print(f'Top interactions: {list(history.index)}')
    recs = hybrid_recommend(user_id, train, top_n=5,
                            alpha=BEST_ALPHA, beta=BEST_BETA, gamma=BEST_GAMMA)
    display(recs[['item_title', 'final_score', 'cf_score', 'content_score', 'pop_score']])
    print()

## 7. Save Final Config

In [ ]:
final_config = {
    'best_alpha': BEST_ALPHA,
    'best_beta':  BEST_BETA,
    'best_gamma': BEST_GAMMA,
    'evaluation': tuning_df.to_dict('records'),
}

with open('../data/processed/hybrid_model_config.pkl', 'wb') as f:
    pickle.dump(final_config, f)

print('Saved hybrid_model_config.pkl')
print('\nRecommendation system complete!')
print('You can now start the FastAPI server: python -m uvicorn main:app --reload --port 8000')